In [21]:
"""
NIR_pole_order.py
======================
Порядок полюса для ОДУ и систем ОДУ методом доминантного баланса.

Поддерживаемые классы
─────────────────────
Одно уравнение:
  * Полиномиальная нелинейность y^n  (n ∈ ℤ, ℚ, ℝ; n отрицательное → kind='zero')
  * Дробные степени y^{m/n}  → при нецелом p возвращает kind='branch_point'
  * Рациональная нелинейность P(y)/Q(y)  (ведущий член при y → ∞)
  * Производные внутри нелинейности: y·y', (y'')², y³·y'
  * Смешанные мономы: y^a·(y')^b  → ξ^{−a·p − b·(p+1)}
  * Неявные уравнения Eq(f, g)  → автоматически lhs−rhs
  * Диссипация/источники: c·y' субдоминантен при p > 1, не влияет на p
  * exp(α·y)  → логарифмический анзац y = −p·log(ξ)
  * tanh(y), atan(y) и др. ограниченные  → заменяются на 1
  * sin(y), cos(y), sinh(y), log(y)  → нет алгебраического полюса
  * Коэффициенты зависящие от z:  z·y'' + ... → замена z → z0+ξ, контроль z0
  * Фиксированные особенности: коэф. имеет полюс в z0 → kind='fixed_singularity'

Система N уравнений:
  * Анзац yi ~ Ai·ξ^{-pi}, решение системы на вектор p

Классификация kind
─────────────────────────────────
  'pole'             p ∈ ℤ>0  — настоящий полюс
  'branch_point'     p ∈ ℚ\ℤ, p>0  — алгебраическая точка ветвления
  'logarithmic'      особенность через exp(α·y)
  'zero'             p < 0  — функция стремится к 0
  'fixed_singularity' коэффициент уравнения сингулярен в z0
  'essential'        sin/cos/sinh при y → ∞
  'log_correction'   log(y) как субдоминантный член
  'none'             нет особенности (линейный ведущий баланс)
  'unknown'          баланс не найден
"""

"\nfind_pole_order.py  v5\n======================\nПорядок полюса для ОДУ и систем ОДУ методом доминантного баланса.\n\nПоддерживаемые классы\n─────────────────────\nОдно уравнение:\n  ✓ Полиномиальная нелинейность y^n  (n ∈ ℤ, ℚ, ℝ; n отрицательное → kind='zero')\n  ✓ Дробные степени y^{m/n}  → при нецелом p возвращает kind='branch_point'\n  ✓ Рациональная нелинейность P(y)/Q(y)  (ведущий член при y → ∞)\n  ✓ Производные внутри нелинейности: y·y', (y'')², y³·y'\n  ✓ Смешанные мономы: y^a·(y')^b  → ξ^{−a·p − b·(p+1)}\n  ✓ Неявные уравнения Eq(f, g)  → автоматически lhs−rhs\n  ✓ Диссипация/источники: c·y' субдоминантен при p > 1, не влияет на p\n  ✓ exp(α·y)  → логарифмический анзац y = −p·log(ξ)\n  ✓ tanh(y), atan(y) и др. ограниченные  → заменяются на 1\n  ✓ sin(y), cos(y), sinh(y), log(y)  → нет алгебраического полюса\n  ✓ Коэффициенты зависящие от z:  z·y'' + ... → замена z → z0+ξ, контроль z0\n  ✓ Фиксированные особенности: коэф. имеет полюс в z0 → kind='fixed_singularity'\n\nСисте

In [22]:
import sympy as sp
from itertools import combinations
from typing import Optional, Tuple, List, Dict

In [23]:
# ══════════════════════════════════════════════════════════════════
#  1. Низкоуровневые операции с показателями ξ
# ══════════════════════════════════════════════════════════════════

def _xi_exp_of_monomial(term: sp.Expr, xi: sp.Symbol) -> sp.Expr:
    """
    Показатель ξ в мономе C·ξ^e.
    ВАЖНО: использует == (не is) — SymPy может пересоздавать объект символа
    с теми же assumptions, но другим id в памяти.
    """
    term = sp.powdenest(term.rewrite(sp.Pow), force=True)
    for node in sp.preorder_traversal(term):
        if node.is_Pow and node.base == xi:
            return node.exp
        if isinstance(node, sp.Symbol) and node == xi:   # == не is
            return sp.Integer(1)
    return sp.Integer(0)


def _dominant_exp(expr: sp.Expr, xi: sp.Symbol, p_syms: list) -> sp.Expr:
    """Минимальный (наиболее сингулярный при ξ→0) показатель ξ в сумме мономов."""
    terms = sp.Add.make_args(sp.expand(expr.rewrite(sp.Pow)))
    exps = []
    for t in terms:
        if sp.simplify(t) == 0:
            continue
        exps.append(_xi_exp_of_monomial(t, xi))
    if not exps:
        return sp.Integer(0)

    def num_key(e):
        try:
            return float(e.subs([(p, 1) for p in p_syms]))
        except Exception:
            return 0.0

    return min(exps, key=num_key)


def _has_essential_xi(expr: sp.Expr, xi: sp.Symbol) -> bool:
    """True если expr содержит exp(f(ξ)) где f зависит от ξ — существенная особенность."""
    for node in sp.preorder_traversal(expr):
        if isinstance(node, sp.exp) and node.args[0].has(xi):
            return True
    return False


def leading_power(expr: sp.Expr, xi: sp.Symbol, p_syms: list) -> Optional[sp.Expr]:
    """
    Ведущий показатель ξ в дробно-рациональном выражении.
    Возвращает exp_num_min − exp_den_min.
    Пропускает термы с существенными особенностями exp(f(ξ)).
    """
    expr = sp.cancel(sp.expand(expr))
    if expr == 0:
        return None
    if _has_essential_xi(expr, xi):
        return None
    num, den = sp.fraction(expr)
    e_num = _dominant_exp(num, xi, p_syms)
    e_den = _dominant_exp(den, xi, p_syms)
    return sp.simplify(e_num - e_den)

In [24]:
# ══════════════════════════════════════════════════════════════════
#  2. Классификация трансцендентных функций от y
# ══════════════════════════════════════════════════════════════════

_BOUNDED   = (sp.tanh, sp.atan, sp.asin, sp.acos, sp.atanh)
_OSCILLATE = (sp.sin, sp.cos, sp.tan)
_ESSENTIAL = (sp.sinh, sp.cosh)


def _classify_transc(expr: sp.Expr, y_funcs: list) -> dict:
    """Определяет какие трансцендентные функции применены к y."""
    res = dict(exp=False, bounded=False, oscillating=False, essential=False,
               log_of_y=False)
    for node in sp.preorder_traversal(expr):
        if not node.args:
            continue
        arg = node.args[0]
        if not any(arg.has(yf) for yf in y_funcs):
            continue
        if isinstance(node, sp.exp):
            res['exp'] = True
        elif isinstance(node, _BOUNDED):
            res['bounded'] = True
        elif isinstance(node, _OSCILLATE):
            res['oscillating'] = True
        elif isinstance(node, _ESSENTIAL):
            res['essential'] = True
        elif isinstance(node, sp.log):
            res['log_of_y'] = True
    return res


def _replace_bounded(expr: sp.Expr, y_funcs: list) -> sp.Expr:
    """tanh(f(y)) → 1, atan(f(y)) → 1 и т.п. (ведущий член при y → ∞)."""
    result = expr
    for node in sp.preorder_traversal(expr):
        if not node.args:
            continue
        if isinstance(node, _BOUNDED) and any(node.args[0].has(yf) for yf in y_funcs):
            result = result.subs(node, sp.Integer(1))
    return result

In [25]:
# ══════════════════════════════════════════════════════════════════
#  3. Фиксированные особенности коэффициентов
# ══════════════════════════════════════════════════════════════════

def _substitute_z0_in_coefficients(
    expr: sp.Expr, z: sp.Symbol, z0: sp.Expr, y_func
) -> Tuple[sp.Expr, str]:
    """
    Подставляет z = z0 в коэффициенты уравнения, не трогая Derivative и y_func.

    Это позволяет анализировать особенности в произвольной точке z0:
      - Регулярный коэффициент в z0 → просто подставляем число
      - Коэффициент → 0 в z0 → терм исчезает (субдоминантен)
      - Коэффициент → ∞ в z0 → фиксированная особенность

    Returns (новое_выражение, 'ok' | 'fixed_singularity')
    """
    result = sp.Integer(0)
    for term in sp.Add.make_args(sp.expand(expr)):
        dep_part  = sp.Integer(1)
        coef_part = sp.Integer(1)
        for factor in sp.Mul.make_args(term):
            if factor.has(sp.Derivative) or factor.has(y_func.func):
                dep_part *= factor
            else:
                coef_part *= factor
        try:
            coef_val = sp.limit(coef_part, z, z0)
        except Exception:
            coef_val = coef_part.subs(z, z0)
        if coef_val in (sp.oo, -sp.oo, sp.zoo, sp.nan):
            return expr, 'fixed_singularity'
        result += coef_val * dep_part
    return sp.expand(result), 'ok'

In [26]:
# ══════════════════════════════════════════════════════════════════
#  4. Подстановка анзаца
# ══════════════════════════════════════════════════════════════════

def _max_deriv_order(expr: sp.Expr, y_func) -> int:
    max_ord = 0
    for node in sp.preorder_traversal(expr):
        if isinstance(node, sp.Derivative) and node.args[0].func == y_func.func:
            ord_ = sum(cnt for _, cnt in node.args[1:])
            max_ord = max(max_ord, ord_)
    return max_ord


def _term_exponents_algebraic(
    expr, primary_y, z, xi, p_syms, A_map, verbose
) -> list:
    """
    Подставляет yi ~ Ai·ξ^{-pi} в expr, возвращает список показателей ξ.
    Обрабатывает: полином, рациональная нелин-ть, дробные степени,
                  ограниченные трансцендентные (→ 1).
    """
    y_funcs = list(A_map.keys())
    expr = _replace_bounded(expr, y_funcs)

    def falling(start, k):
        r = sp.Integer(1)
        for i in range(k):
            r *= (start - i)
        return r

    # Строим таблицу подстановок
    deriv_subs = []
    func_subs  = []
    for yf, A_sym in A_map.items():
        p_sym  = p_syms[list(A_map.keys()).index(yf)]
        max_ord = _max_deriv_order(expr, yf)
        for k in range(1, max_ord + 1):
            key = sp.Derivative(yf, (z, k))
            val = A_sym * falling(-p_sym, k) * xi**(-p_sym - k)
            deriv_subs.append((key, val, k))
        func_subs.append((yf, A_sym * xi**(-p_sym)))

    # Применяем: производные по убыванию порядка, потом функции
    result = expr
    for key, val, _ in sorted(deriv_subs, key=lambda x: -x[2]):
        result = result.subs(key, val)
    for key, val in func_subs:
        result = result.subs(key, val)

    result = sp.expand(result.rewrite(sp.Pow))

    exponents = []
    p_list = list(p_syms)
    for term in sp.Add.make_args(sp.expand(result)):
        term_c = sp.cancel(sp.powdenest(sp.expand(term), force=True))
        if sp.simplify(term_c) == 0:
            continue
        e = leading_power(term_c, xi, p_list)
        if e is not None:
            e = sp.simplify(e)
            exponents.append(e)
            if verbose:
                print(f"    [alg] {str(sp.simplify(term_c)):55s}  ξ^({e})")
    return exponents


def _term_exponents_log(expr, y_func, z, xi, p_sym, verbose) -> list:
    """
    Логарифмический анзац y = −p·log(ξ):
      y^(k) = (−1)^k·(k−1)!·p·ξ^{−k}
      exp(α·y) → ξ^{−α·p}
    """
    max_ord = _max_deriv_order(expr, y_func)
    y_sub   = -p_sym * sp.log(xi)

    subs_d = {
        sp.Derivative(y_func, (z, k)):
            (-1)**k * sp.factorial(k - 1) * p_sym * xi**(-k)
        for k in range(1, max_ord + 1)
    }

    result = expr
    for k in range(max_ord, 0, -1):
        key = sp.Derivative(y_func, (z, k))
        if result.has(key):
            result = result.subs(key, subs_d[key])
    result = result.subs(y_func, y_sub)
    result = sp.powdenest(sp.expand(result.rewrite(sp.Pow)), force=True)

    exponents = []
    for term in sp.Add.make_args(sp.expand(result)):
        term_c = sp.cancel(sp.powdenest(sp.expand(term), force=True))
        if sp.simplify(term_c) == 0:
            continue
        e = leading_power(term_c, xi, [p_sym])
        if e is not None:
            e = sp.simplify(e)
            exponents.append(e)
            if verbose:
                print(f"    [log] ξ^({e})")
    return exponents

In [27]:
# ══════════════════════════════════════════════════════════════════
#  5. Баланс показателей
# ══════════════════════════════════════════════════════════════════

def _balance_single(exponents: list, p_sym,
                    allow_nonpositive: bool = False) -> Optional[sp.Expr]:
    """
    Одна переменная p.
    Возвращает МАКСИМАЛЬНЫЙ валидный p (наиболее сингулярный баланс).
    Это соответствует методу Пейнлеве: берём старший доминирующий член.

    При allow_nonpositive=True используем символ без positive=True,
    иначе SymPy блокирует отрицательные решения на уровне solve().
    """
    # Если нужны отрицательные p — заменяем на символ без ограничений
    if allow_nonpositive and getattr(p_sym, 'is_positive', False):
        p_free = sp.Symbol('_p_free', real=True)
        exponents = [e.subs(p_sym, p_free) for e in exponents]
        solve_sym = p_free
    else:
        solve_sym = p_sym

    candidates = set()
    for e1, e2 in combinations(exponents, 2):
        diff = sp.simplify(e1 - e2)
        if diff == 0:
            continue
        try:
            sols = sp.solve(diff, solve_sym, dict=True)
        except Exception:
            continue
        for s in sols:
            val = s.get(solve_sym)
            if val is None:
                continue
            val = sp.simplify(val)
            if not val.is_real:
                continue
            fval = float(val)
            if not allow_nonpositive and fval <= 1e-12:
                continue
            candidates.add(val)

    valid = []
    for cand in sorted(candidates, key=float):
        try:
            evals = [float(e.subs(solve_sym, cand)) for e in exponents]
        except Exception:
            continue
        min_v = min(evals)
        if sum(1 for v in evals if abs(v - min_v) < 1e-9) >= 2:
            valid.append(cand)

    if not valid:
        return None
    return max(valid, key=float)


def _balance_system(eq_exponents_list: list, p_syms: list) -> Optional[dict]:
    """
    Система N уравнений, N переменных.
    Из каждого уравнения: два наиболее сингулярных терма приравниваются.
    """
    balance_eqs = []
    for exps in eq_exponents_list:
        if len(exps) < 2:
            continue
        def num_key(e):
            try:
                return float(e.subs([(ps, 1) for ps in p_syms]))
            except Exception:
                return 0.0
        sorted_exps = sorted(exps, key=num_key)
        balance_eqs.append(sp.Eq(sorted_exps[0], sorted_exps[1]))

    if not balance_eqs:
        return None
    try:
        sol = sp.solve(balance_eqs, p_syms, dict=True)
    except Exception:
        return None
    if not sol:
        return None

    sol_list = sol if isinstance(sol, list) else [sol]
    valid = []
    for s in sol_list:
        vals = {ps: s.get(ps) for ps in p_syms}
        if any(v is None for v in vals.values()):
            continue
        if all(sp.simplify(v).is_positive for v in vals.values()):
            valid.append(vals)

    if not valid:
        return None

    def norm(d):
        try:
            return sum(float(v)**2 for v in d.values())
        except Exception:
            return float('inf')

    return min(valid, key=norm)

In [28]:
# ══════════════════════════════════════════════════════════════════
#  6. Классификация p → kind
# ══════════════════════════════════════════════════════════════════

def _classify_p(p_val: sp.Expr) -> str:
    """
    Уточняет kind на основе значения p:
      p ∈ ℤ>0  → 'pole'
      p ∈ ℚ\ℤ  → 'branch_point'
      p < 0    → 'zero'
    """
    if p_val is None:
        return 'unknown'
    try:
        fval = float(p_val)
    except Exception:
        return 'algebraic'
    if fval < -1e-12:
        return 'zero'
    p_rat = sp.nsimplify(p_val, rational=True, tolerance=1e-10)
    if p_rat.is_integer or (p_rat.is_Rational and p_rat.q == 1):
        return 'pole'
    return 'branch_point'

In [29]:
# ══════════════════════════════════════════════════════════════════
#  7. Публичный API
# ══════════════════════════════════════════════════════════════════

def find_pole_order(
    eq,
    y_func: sp.Function,
    z: sp.Symbol,
    *,
    z0: sp.Expr = None,
    allow_nonpositive: bool = False,
    verbose: bool = False,
) -> Tuple[Optional[sp.Expr], str]:
    """
    Порядок полюса для одного ОДУ.

    Parameters
    ----------
    eq               : уравнение (Eq или выражение = 0)
    y_func           : функция y(z)
    z                : независимая переменная
    z0               : точка особенности (по умолчанию 0).
                       Если передать z0, производится замена z → z0+ξ.
    allow_nonpositive: возвращать нули (p ≤ 0 → kind='zero')
    verbose          : промежуточные шаги

    Returns
    -------
    (p, kind)
      kind: 'pole' | 'branch_point' | 'logarithmic' | 'zero'
            'fixed_singularity' | 'essential' | 'log_correction'
            'none' | 'unknown'
    """
    expr = (eq.lhs - eq.rhs) if isinstance(eq, sp.Eq) else sp.expand(eq)
    expr = sp.expand(expr)

    # Замена z → z0+ξ если задана точка особенности
    xi = sp.Symbol('xi', positive=True)
    p  = sp.Symbol('p',  positive=True, real=True)
    A  = sp.Symbol('A',  positive=True)

    if z0 is not None:
        if verbose:
            print(f"[info] Подстановка z = {z0} в коэффициенты")
        expr, status = _substitute_z0_in_coefficients(expr, z, z0, y_func)
        if status == 'fixed_singularity':
            return None, 'fixed_singularity'

    transc = _classify_transc(expr, [y_func])
    if verbose and any(transc.values()):
        active = [k for k, v in transc.items() if v]
        print(f"[info] Трансцендентные функции от y: {active}")

    # ── Алгебраический анзац ───────────────────────────────────
    exps = _term_exponents_algebraic(
        expr, y_func, z, xi, [p], {y_func: A}, verbose
    )

    if len(exps) >= 2:
        p_val = _balance_single(exps, p, allow_nonpositive=allow_nonpositive)
        if p_val is not None:
            return p_val, _classify_p(p_val)

    # ── Логарифмический анзац ──────────────────────────────────
    if transc['exp']:
        log_exps = _term_exponents_log(expr, y_func, z, xi, p, verbose)
        if len(log_exps) >= 2:
            p_log = _balance_single(log_exps, p, allow_nonpositive=False)
            if p_log is not None:
                return p_log, 'logarithmic'

    # ── Диагностика по типу ───────────────────────────────────
    if transc['oscillating']:
        return None, 'essential'
    if transc['essential']:
        return None, 'essential'
    if transc['log_of_y']:
        return None, 'log_correction'
    if transc['bounded'] and not transc['exp']:
        return None, 'none'

    return None, 'unknown'


def find_pole_order_system(
    equations: list,
    y_funcs: list,
    z: sp.Symbol,
    *,
    z0: sp.Expr = None,
    verbose: bool = False,
) -> Tuple[Optional[dict], str]:
    """
    Порядок полюса для системы ОДУ.

    Returns ({y_func: p_value, ...}, kind)
    """
    if len(equations) != len(y_funcs):
        raise ValueError("len(equations) должно совпадать с len(y_funcs)")

    xi     = sp.Symbol('xi', positive=True)
    N      = len(y_funcs)
    p_syms = [sp.Symbol(f'p{i+1}', positive=True) for i in range(N)]
    A_map  = {yf: sp.Symbol(f'A{i+1}', positive=True) for i, yf in enumerate(y_funcs)}

    exprs = []
    for eq in equations:
        e = (eq.lhs - eq.rhs) if isinstance(eq, sp.Eq) else sp.expand(eq)
        if z0 is not None:
            e = sp.expand(e.subs(z, z0 + xi))
        exprs.append(sp.expand(e))

    all_exponents = []
    for i, expr in enumerate(exprs):
        if verbose:
            print(f"\n[Уравнение {i+1}]")
        exps = _term_exponents_algebraic(
            expr, y_funcs[i], z, xi, p_syms, A_map, verbose
        )
        all_exponents.append(exps)

    sol = _balance_system(all_exponents, p_syms)
    if sol is None:
        return None, 'unknown'

    result   = {y_funcs[i]: sol[p_syms[i]] for i in range(N)}
    p_kinds  = [_classify_p(sol[p_syms[i]]) for i in range(N)]
    # Если все p — целые → 'pole', иначе смешанный → 'branch_point'
    kind = 'pole' if all(k == 'pole' for k in p_kinds) else 'branch_point'
    return result, kind

In [30]:
"""
test_pole_order.py  v2
======================
Полный набор тестов для find_pole_order.py v5.

Разделы
───────
§1   Пейнлеве I–IV
§2   Математическая физика (KdV, Burgers, NLS, KS, Fisher, ...)
§3   Курамото–Сивашинский (все формы, включая форму с картинки)
§4   Чейзи
§5   Уравнение Абеля
§6   Реакция–диффузия / бегущие волны
§7   Диссипация и источники (общий случай)
§8   Полиномиальная нелинейность: таблица y^{(k)}=y^n
§9   Дробные степени → branch_point
§10  Нелинейное затухание y^n·y'
§11  Рациональная нелинейность P(y)/Q(y)
§12  Неявные уравнения (y')²=f(y)
§13  Конкурирующие балансы y^m+y^n
§14  Смешанные мономы y^a·(y')^b
§15  Трансцендентные функции от y
§16  Отрицательные степени → нули (allow_nonpositive)
§17  Переменные коэффициенты (z0)
§18  Вырожденные / нет полюса
§19  Системы уравнений
"""

z  = sp.Symbol('z')
y  = sp.Function('y')
w  = sp.Function('w')
nu = sp.Symbol('nu',    positive=True)
sigma = sp.Symbol('sigma',    positive=True)
al = sp.Symbol('alpha', real=True)
be = sp.Symbol('beta',  positive=True)
a, b, g, d = [sp.Symbol(s) for s in ('a','b','g','d')]

R = sp.Rational
I = sp.Integer

# ══════════════════════════════════════════════════════════════
#  Одиночные уравнения: (описание, eq, exp_p, exp_kind, kwargs)
# ══════════════════════════════════════════════════════════════

SINGLE = [

    # §1 Пейнлеве
    ("P-I:    y''=6y²+z",
     sp.Eq(y(z).diff(z,2), 6*y(z)**2+z),                             I(2), 'pole', {}),
    ("P-II:   y''=2y³+zy+α",
     sp.Eq(y(z).diff(z,2), 2*y(z)**3+z*y(z)+al),                     I(1), 'pole', {}),
    ("P-III:  y''=(y')²/y−y'/z+αy²/z+β/z+γy³+δ/y",
     sp.Eq(y(z).diff(z,2), y(z).diff(z)**2/y(z) - y(z).diff(z)/z
           + a*y(z)**2/z + b/z + g*y(z)**3 + d/y(z)),                I(1), 'pole', {}),
    ("P-IV:   y''=(y')²/(2y)+3y³/2+4zy²+2(z²−α)y−β/(2y)",
     sp.Eq(y(z).diff(z,2), y(z).diff(z)**2/(2*y(z)) + R(3,2)*y(z)**3
           + 4*z*y(z)**2 + 2*(z**2-al)*y(z) - be/(2*y(z))),          I(1), 'pole', {}),

    # §2 Матфизика
    ("KdV стац:  y'''+y·y'=0",
     sp.Eq(y(z).diff(z,3)+y(z)*y(z).diff(z), 0),                     I(2), 'pole', {}),
    ("Burgers:   y'+y·y'=ν·y''",
     sp.Eq(y(z).diff(z)+y(z)*y(z).diff(z), nu*y(z).diff(z,2)),       I(1), 'pole', {}),
    ("NLS стац:  y''=y³",
     sp.Eq(y(z).diff(z,2), y(z)**3),                                  I(1), 'pole', {}),
    ("NLS стац:  y''=y³−y",
     sp.Eq(y(z).diff(z,2), y(z)**3-y(z)),                             I(1), 'pole', {}),
    ("Гинзбург–Ландау: y''=y−y³",
     sp.Eq(y(z).diff(z,2), y(z)-y(z)**3),                            I(1), 'pole', {}),
    ("Дюффинг:   y''+αy+βy³=0",
     sp.Eq(y(z).diff(z,2)+al*y(z)+be*y(z)**3, 0),                    I(1), 'pole', {}),

    # §3 Курамото–Сивашинский
    # Стандартный (u_t + u*u_x + u_xx + u_xxxx = 0, бегущая волна)
    ("KS стандарт: y''''+y''+y·y'=0",
     sp.Eq(y(z).diff(z,4)+y(z).diff(z,2)+y(z)*y(z).diff(z), 0),     I(3), 'pole', {}),
    # Форма: u_t + u*u_x + u_xx + sigma*u_xxx + u_xxxx = 0
    ("KS (картинка): y''''+sigma*y'''+y''+y·y'−cy'=0",
     sp.Eq(y(z).diff(z,4)+6*y(z).diff(z,3)+y(z).diff(z,2)
           -sp.Symbol('c')*y(z).diff(z)+y(z)*y(z).diff(z), 0),       I(3), 'pole', {}),
    ("KS без y'': y''''+y·y'=0",
     sp.Eq(y(z).diff(z,4)+y(z)*y(z).diff(z), 0),                     I(3), 'pole', {}),
    ("KS интегр:  y'''+y''+y²/2=0",
     sp.Eq(y(z).diff(z,3)+y(z).diff(z,2)+y(z)**2/2, 0),              I(3), 'pole', {}),

    # §4 Чейзи
    ("Чейзи: y'''=2y·y''−3(y')²",
     sp.Eq(y(z).diff(z,3), 2*y(z)*y(z).diff(z,2)-3*y(z).diff(z)**2),I(1), 'pole', {}),

    # §5 Абель
    ("Абель: y'=y³+y²",   sp.Eq(y(z).diff(z), y(z)**3+y(z)**2),   R(1,2),'branch_point',{}),
    ("Абель: y'=y³−y",    sp.Eq(y(z).diff(z), y(z)**3-y(z)),       R(1,2),'branch_point',{}),
    ("Абель: y'=y³",      sp.Eq(y(z).diff(z), y(z)**3),            R(1,2),'branch_point',{}),

    # §6 Реакция–диффузия
    ("Fisher:   y''+y'=y−y²",
     sp.Eq(y(z).diff(z,2)+y(z).diff(z), y(z)-y(z)**2),               I(2), 'pole', {}),
    ("KPP:      y''=y·(1−y)",
     sp.Eq(y(z).diff(z,2), y(z)*(1-y(z))),                            I(2), 'pole', {}),

    # §7 Диссипация/источники — p не зависит от коэффициента при y'
    ("y''+0.5y'=y²",    sp.Eq(y(z).diff(z,2)+R(1,2)*y(z).diff(z), y(z)**2),    I(2),'pole',{}),
    ("y''+10y' =y²",    sp.Eq(y(z).diff(z,2)+10*y(z).diff(z),       y(z)**2),    I(2),'pole',{}),
    ("y''−y'   =y²",    sp.Eq(y(z).diff(z,2)-y(z).diff(z),          y(z)**2),    I(2),'pole',{}),
    ("y''+y'   =y³",    sp.Eq(y(z).diff(z,2)+y(z).diff(z),          y(z)**3),    I(1),'pole',{}),
    ("y''+y'   =y⁴",    sp.Eq(y(z).diff(z,2)+y(z).diff(z),          y(z)**4),  R(2,3),'branch_point',{}),
    ("y''+y    =y²",    sp.Eq(y(z).diff(z,2)+y(z),                   y(z)**2),    I(2),'pole',{}),
    ("y''+y    =y³",    sp.Eq(y(z).diff(z,2)+y(z),                   y(z)**3),    I(1),'pole',{}),
    ("y'''+y''+y'=y²",  sp.Eq(y(z).diff(z,3)+y(z).diff(z,2)+y(z).diff(z),y(z)**2),I(3),'pole',{}),

    # §8 Таблица y^{(k)} = y^n
    ("y' =y², p=1",  sp.Eq(y(z).diff(z),   y(z)**2), I(1),'pole',{}),
    ("y''=y², p=2",  sp.Eq(y(z).diff(z,2), y(z)**2), I(2),'pole',{}),
    ("y³'=y², p=3",  sp.Eq(y(z).diff(z,3), y(z)**2), I(3),'pole',{}),
    ("y''''=y², p=4",sp.Eq(y(z).diff(z,4), y(z)**2), I(4),'pole',{}),
    ("y⁵=y², p=5",   sp.Eq(y(z).diff(z,5), y(z)**2), I(5),'pole',{}),
    ("y' =y³, p=1/2",sp.Eq(y(z).diff(z),   y(z)**3), R(1,2),'branch_point',{}),
    ("y''=y³, p=1",  sp.Eq(y(z).diff(z,2), y(z)**3), I(1),'pole',{}),
    ("y³'=y³, p=3/2",sp.Eq(y(z).diff(z,3), y(z)**3), R(3,2),'branch_point',{}),
    ("y''''=y³,p=2", sp.Eq(y(z).diff(z,4), y(z)**3), I(2),'pole',{}),
    ("y''=y⁴, p=2/3",sp.Eq(y(z).diff(z,2), y(z)**4), R(2,3),'branch_point',{}),
    ("y³'=y⁴, p=1",  sp.Eq(y(z).diff(z,3), y(z)**4), I(1),'pole',{}),
    ("y''''=y⁴,p=4/3",sp.Eq(y(z).diff(z,4),y(z)**4),R(4,3),'branch_point',{}),
    ("y''''=y², p=4",sp.Eq(y(z).diff(z,4), y(z)**2), I(4),'pole',{}),
    ("y⁵=y², p=5",   sp.Eq(y(z).diff(z,5), y(z)**2), I(5),'pole',{}),

    # §9 Дробные степени
    ("y'=y^{3/2},  p=2",  sp.Eq(y(z).diff(z),y(z)**R(3,2)),  I(2),  'pole',{}),
    ("y'=y^{4/3},  p=3",  sp.Eq(y(z).diff(z),y(z)**R(4,3)),  I(3),  'pole',{}),
    ("y'=y^{5/3},  p=3/2",sp.Eq(y(z).diff(z),y(z)**R(5,3)),  R(3,2),'branch_point',{}),
    ("y'=y^{5/4},  p=4",  sp.Eq(y(z).diff(z),y(z)**R(5,4)),  I(4),  'pole',{}),
    ("y'=y^{7/4},  p=4/3",sp.Eq(y(z).diff(z),y(z)**R(7,4)),  R(4,3),'branch_point',{}),
    ("y'=y^4,      p=1/3",sp.Eq(y(z).diff(z),y(z)**4),        R(1,3),'branch_point',{}),
    ("y'=y^5,      p=1/4",sp.Eq(y(z).diff(z),y(z)**5),        R(1,4),'branch_point',{}),

    # §10 Нелинейное затухание y^n·y'
    ("y''=y·y',  p=1",  sp.Eq(y(z).diff(z,2),y(z)*y(z).diff(z)),   I(1),  'pole',{}),
    ("y''=y²·y', p=1/2",sp.Eq(y(z).diff(z,2),y(z)**2*y(z).diff(z)),R(1,2),'branch_point',{}),
    ("y''=y³·y', p=1/3",sp.Eq(y(z).diff(z,2),y(z)**3*y(z).diff(z)),R(1,3),'branch_point',{}),

    # §11 Рациональная нелинейность
    ("y''=y⁴/(1+y²) → p=2",
     sp.Eq(y(z).diff(z,2), y(z)**4/(1+y(z)**2)),                      I(2),'pole',{}),
    ("y''=y³/(1+y²) → нет (ведущий~y, линейно)",
     sp.Eq(y(z).diff(z,2), y(z)**3/(1+y(z)**2)),                      None,'unknown',{}),

    # §12 Неявные: (y')²=f(y)
    ("(y')²=y³,         p=2", sp.Eq(y(z).diff(z)**2, y(z)**3),            I(2),'pole',{}),
    ("(y')²=y³+y²,      p=2", sp.Eq(y(z).diff(z)**2, y(z)**3+y(z)**2),    I(2),'pole',{}),
    ("(y')²=y²(1−y²),   p=1", sp.Eq(y(z).diff(z)**2, y(z)**2*(1-y(z)**2)),I(1),'pole',{}),
    ("(y')²=y⁴−1,       p=1", sp.Eq(y(z).diff(z)**2, y(z)**4-1),          I(1),'pole',{}),
    ("(y')²=y⁴,         p=1", sp.Eq(y(z).diff(z)**2, y(z)**4),            I(1),'pole',{}),

    # §13 Конкурирующие балансы — побеждает БОЛЕЕ сингулярный
    ("y''=y²+y³  → p=1  (y³ доминирует)",
     sp.Eq(y(z).diff(z,2), y(z)**2+y(z)**3),                          I(1),'pole',{}),
    ("y''=y³+y⁴  → p=2/3 (y⁴ доминирует)",
     sp.Eq(y(z).diff(z,2), y(z)**3+y(z)**4),                        R(2,3),'branch_point',{}),
    ("y'=y²+y³   → p=1/2 (y³ доминирует)",
     sp.Eq(y(z).diff(z), y(z)**2+y(z)**3),                           R(1,2),'branch_point',{}),

    # §14 Смешанные мономы
    ("y''=(y')²    → нет полюса (p=0 невалидно)",
     sp.Eq(y(z).diff(z,2), y(z).diff(z)**2),                          None,'unknown',{}),
    ("y·y''=(y')²  → вырождение",
     sp.Eq(y(z)*y(z).diff(z,2), y(z).diff(z)**2),                     None,'unknown',{}),

    # §15 Трансцендентные
    ("y'=exp(y)          → логариф. p=1",
     sp.Eq(y(z).diff(z), sp.exp(y(z))),                                I(1),'logarithmic',{}),
    ("y''=tanh(y)·y²     → tanh→1, p=2",
     sp.Eq(y(z).diff(z,2), sp.tanh(y(z))*y(z)**2),                    I(2),'pole',{}),
    ("y''=sin(y)+y²      → y² доминирует, p=2",
     sp.Eq(y(z).diff(z,2), sp.sin(y(z))+y(z)**2),                     I(2),'pole',{}),
    ("y'=sin(y)           → essential",
     sp.Eq(y(z).diff(z), sp.sin(y(z))),                                None,'essential',{}),
    ("Маятник: y''+sin(y)=0 → essential",
     sp.Eq(y(z).diff(z,2)+sp.sin(y(z)), 0),                           None,'essential',{}),
    ("y'=log(y)           → log_correction",
     sp.Eq(y(z).diff(z), sp.log(y(z))),                               None,'log_correction',{}),

    # §16 Отрицательные степени → нули (p<0)
    ("y'=y^{−1},   p=−1/2",sp.Eq(y(z).diff(z),1/y(z)),              R(-1,2),'zero',{'allow_nonpositive':True}),
    ("y''=y^{−2},  p=−2/3",sp.Eq(y(z).diff(z,2),y(z)**(-2)),        R(-2,3),'zero',{'allow_nonpositive':True}),
    ("y'=y^{1/3},  p=−3/2",sp.Eq(y(z).diff(z),y(z)**R(1,3)),        R(-3,2),'zero',{'allow_nonpositive':True}),

    # §17 Переменные коэффициенты
    ("z·y''=y²  в z0=1  → p=2",
     sp.Eq(z*y(z).diff(z,2), y(z)**2),                                 I(2),'pole',{'z0':I(1)}),
    ("(z−1)·y''=y³  в z0=2  → p=1",
     sp.Eq((z-1)*y(z).diff(z,2), y(z)**3),                             I(1),'pole',{'z0':I(2)}),

    # §18 Вырожденные
    ("y''=y³/(1+y²) → нет",
     sp.Eq(y(z).diff(z,2),y(z)**3/(1+y(z)**2)),                       None,'unknown',{}),
    ("y'=sin(y)      → essential",
     sp.Eq(y(z).diff(z),sp.sin(y(z))),                                 None,'essential',{}),
    ("y'=log(y)      → log_correction",
     sp.Eq(y(z).diff(z),sp.log(y(z))),                                 None,'log_correction',{}),
]

# ══════════════════════════════════════════════════════════════
#  Системы: (описание, [eq], [func], {ожид}, kind, kwargs)
# ══════════════════════════════════════════════════════════════

SYSTEM = [
    ("y'=w, w'=y²    → p_y=2, p_w=3",
     [sp.Eq(y(z).diff(z),w(z)), sp.Eq(w(z).diff(z),y(z)**2)],
     [y(z),w(z)], {y(z):I(2),w(z):I(3)}, 'pole', {}),

    ("y'=w, w'=y³    → p_y=1, p_w=2",
     [sp.Eq(y(z).diff(z),w(z)), sp.Eq(w(z).diff(z),y(z)**3)],
     [y(z),w(z)], {y(z):I(1),w(z):I(2)}, 'pole', {}),

    ("y'=w, w'=y⁴    → p_y=2/3, p_w=5/3  [branch]",
     [sp.Eq(y(z).diff(z),w(z)), sp.Eq(w(z).diff(z),y(z)**4)],
     [y(z),w(z)], {y(z):R(2,3),w(z):R(5,3)}, 'branch_point', {}),

    ("y'=wy, w'=y²−w → p_y=1, p_w=1",
     [sp.Eq(y(z).diff(z),w(z)*y(z)), sp.Eq(w(z).diff(z),y(z)**2-w(z))],
     [y(z),w(z)], {y(z):I(1),w(z):I(1)}, 'pole', {}),
]


# ══════════════════════════════════════════════════════════════
#  Runner
# ══════════════════════════════════════════════════════════════

def run():
    G='\033[92m'; R='\033[91m'; Y='\033[93m'; B='\033[94m'; E='\033[0m'
    passed = total = 0

    # ── §1–§18 одиночные ──────────────────────────────────────
    sections = {
        0:  "§1  Пейнлеве I–IV",
        4:  "§2  Математическая физика",
        10: "§3  Курамото–Сивашинский (+ форма с картинки)",
        14: "§4  Чейзи",
        15: "§5  Абель",
        18: "§6  Реакция–диффузия",
        20: "§7  Диссипация/источники",
        28: "§8  Таблица y^(k)=y^n",
        43: "§9  Дробные степени",
        50: "§10 Нелин. затухание y^n·y'",
        53: "§11 Рациональная нелинейность",
        55: "§12 Неявные (y')²=f(y)",
        60: "§13 Конкурирующие балансы",
        63: "§14 Смешанные мономы",
        65: "§15 Трансцендентные",
        71: "§16 Отрицательные степени",
        74: "§17 Переменные коэффициенты",
        76: "§18 Вырожденные",
    }

    print(f"\n{'═'*72}")
    print(f"{B}ОДИНОЧНЫЕ УРАВНЕНИЯ  ({len(SINGLE)} тестов){E}")
    print(f"{'═'*72}")

    for idx,(desc,eq,exp_p,exp_kind,kw) in enumerate(SINGLE):
        total += 1
        if idx in sections:
            print(f"\n  {Y}{sections[idx]}{E}")
        p_res, kind_res = find_pole_order(eq, y(z), z, **kw)
        ok = (p_res==exp_p) and (kind_res==exp_kind)
        passed += ok
        mark = f"{G}✓{E}" if ok else f"{R}✗{E}"
        label = f"p={p_res}" if p_res is not None else "—"
        print(f"  {mark}  {desc}")
        if not ok:
            print(f"       {R}Получено:  p={p_res}, kind={kind_res}{E}")
            print(f"       Ожидалось: p={exp_p}, kind={exp_kind}")
        else:
            print(f"       → {label}  [{kind_res}]")

    # ── §19 системы ───────────────────────────────────────────
    print(f"\n{'═'*72}")
    print(f"{B}§19  СИСТЕМЫ УРАВНЕНИЙ  ({len(SYSTEM)} тестов){E}")
    print(f"{'═'*72}")

    for desc,eqs,funcs,exp_dict,exp_kind,kw in SYSTEM:
        total += 1
        res_dict, kind_res = find_pole_order_system(eqs, funcs, z, **kw)
        ok = (res_dict is not None and kind_res==exp_kind and
              all(res_dict.get(f)==exp_dict[f] for f in funcs))
        passed += ok
        mark = f"{G}✓{E}" if ok else f"{R}✗{E}"
        print(f"  {mark}  {desc}")
        if not ok:
            print(f"       {R}Получено:  {res_dict}, kind={kind_res}{E}")
            print(f"       Ожидалось: {exp_dict}, kind={exp_kind}")
        else:
            parts=', '.join(f"p_{f}={v}" for f,v in res_dict.items())
            print(f"       → {parts}  [{kind_res}]")

    # ── Итог ──────────────────────────────────────────────────
    color = G if passed==total else (Y if passed/total>0.9 else R)
    pct = 100*passed//total
    print(f"\n{'═'*72}")
    print(f"{color}Пройдено: {passed}/{total}  ({pct}%){E}\n")
    return passed, total

if __name__ == '__main__':
    run()



════════════════════════════════════════════════════════════════════════
ОДИНОЧНЫЕ УРАВНЕНИЯ  (78 тестов)
════════════════════════════════════════════════════════════════════════

  §1  Пейнлеве I–IV
  ✓  P-I:    y''=6y²+z
       → p=2  [pole]
  ✓  P-II:   y''=2y³+zy+α
       → p=1  [pole]
  ✓  P-III:  y''=(y')²/y−y'/z+αy²/z+β/z+γy³+δ/y
       → p=1  [pole]
  ✓  P-IV:   y''=(y')²/(2y)+3y³/2+4zy²+2(z²−α)y−β/(2y)
       → p=1  [pole]

  §2  Математическая физика
  ✓  KdV стац:  y'''+y·y'=0
       → p=2  [pole]
  ✓  Burgers:   y'+y·y'=ν·y''
       → p=1  [pole]
  ✓  NLS стац:  y''=y³
       → p=1  [pole]
  ✓  NLS стац:  y''=y³−y
       → p=1  [pole]
  ✓  Гинзбург–Ландау: y''=y−y³
       → p=1  [pole]
  ✓  Дюффинг:   y''+αy+βy³=0
       → p=1  [pole]

  §3  Курамото–Сивашинский (+ форма с картинки)
  ✓  KS стандарт: y''''+y''+y·y'=0
       → p=3  [pole]
  ✓  KS (картинка): y''''+sigma*y'''+y''+y·y'−cy'=0
       → p=3  [pole]
  ✓  KS без y'': y''''+y·y'=0
       → p=3  [pole]
  ✓  KS интег